In [0]:
from pyspark.sql import functions as F, Window

bronze = spark.table("workspace.bronze.empresas_raw")
print("linhas na bronze:", bronze.count())

In [0]:
# Visão geral da tabela bronze de empresas
print("linhas:", bronze.count(), "| colunas:", len(bronze.columns))
display(bronze)


In [0]:
#descartar colunas irrelevantes para a minha análise (guardei na camada bronze pq pode ser útil por algum motivo) Mas agora na camada silver não é necessário

COLUNAS_RUIDO = ["cor_marca", "codigo_legado", "flag_teste",
                 "obs_interna", "hash_antigo", "versao_layout"]
df = bronze.drop(*COLUNAS_RUIDO)
print(len(bronze.columns), "->", len(df.columns), "colunas")

In [0]:
# Limpeza básica de texto (em todas as colunas, menos metadados que começam com "_"):
#   1) TRIM: remove espaços no começo e no fim
#   2) Nulos disfarçados ("", "N/A", "N/D", "NULL", vistos no perfilamento) viram NULL de verdade

NULOS_DISFARCADOS = ["", "N/A", "N/D", "NULL"]

def limpa_texto(nome_coluna):
    t = F.trim(F.col(nome_coluna))
    return F.when(F.upper(t).isin(NULOS_DISFARCADOS), None).otherwise(t)

for c in df.columns:
    if not c.startswith("_"):
        df = df.withColumn(c, limpa_texto(c))

In [0]:
# Comparar o nulos por coluna depois do cód 4
df.select([F.sum(F.col(c).isNull().cast("int")).alias(c)
           for c in df.columns if not c.startswith("_")]).show(vertical=True)

In [0]:
# Transformar os dados das colunas ticker, situacao e uf tudo em maiúsculas
df = df.withColumn("ticker", F.upper(F.col("ticker")))
df = df.withColumn("situacao", F.upper(F.col("situacao")))
df = df.withColumn("uf", F.upper(F.col("uf")))

#mostrar o resultado
display(df)




In [0]:
#Criar lista pas as colunas "setor", "subsetor", "cidade", "segmento_listagem", "nome_pregao") e padronizar os dados para que fiquem com a primeira letra de cada palavra em maiúscula

for c in ["setor", "subsetor", "cidade", "segmento_listagem", "nome_pregao"]:
    df = df.withColumn(c, F.initcap(F.col(c)))

#mostrar o resultado
display(df)


In [0]:
#Tratar "razao_social". Não vou usar apenas initcap pois vai gerar problema com S.A final. Então vou usar initcap e depois substituir S.a por S.A

df = df.withColumn("razao_social", F.initcap(F.col("razao_social")))
df = df.withColumn("razao_social", F.regexp_replace(F.col("razao_social"), r"S\.a\.", "S.A."))

#mostrar o resultado
display(df)


In [0]:
# Validação: quantidade de valores distintos por coluna padronizada

for c in ["situacao", "setor", "subsetor", "segmento_listagem", "uf", "cidade"]:
    print(c, "->", df.select(c).distinct().count())

# Verificar se está tudo OK. Razões sociais que NÃO terminam em "S.A." (esperado: 0)
df.filter(~F.col("razao_social").endswith("S.A.")).count()

In [0]:
# TIPAGEM 1/4: id_empresa (string -> int)
# Por quê: nas cotações o id vem como '000123' e aqui como '123'. Como texto, o JOIN
# retorna 0 linhas. Com int os dois viram 123 e o JOIN funciona.
# try_cast devolve NULL em vez de derrubar a célula se algum id não for numérico.
df = df.withColumn("id_empresa", F.col("id_empresa").try_cast("int"))

# Validação
df.select("id_empresa").printSchema()                    
# esperado: integer
print(df.filter(F.col("id_empresa").isNull()).count())    
# esperado: 0 (nenhum id perdido na conversão)

In [0]:
# Tratar as duas colunas de datas
# Referência para a reconciliação: quantos "só ano" e quantos nulos existem ANTES da conversão
for c in ["data_fundacao", "data_listagem"]:
    so_ano = df.filter(F.col(c).rlike("^[0-9]{4}$")).count()
    nulos = df.filter(F.col(c).isNull()).count()
    print(c, "| só ano:", so_ano, "| nulos antes:", nulos)

In [0]:
# Datas chegam em 5 formatos. try_to_date devolve NULL se o formato não bater,
# e coalesce fica com o primeiro formato que funcionar.
# Defini a função converte_data  
def converte_data(col):
    return F.coalesce(
        F.expr(f"try_to_date({col}, 'dd/MM/yyyy')"),
        F.expr(f"try_to_date({col}, 'yyyy-MM-dd')"),
        F.expr(f"try_to_date({col}, 'dd-MM-yyyy')"),
        F.expr(f"try_to_date({col}, 'yyyyMMdd')"),
        F.expr(f"try_to_date({col}, 'dd.MM.yyyy')"),
    )

In [0]:
 df = df.withColumn("data_fundacao", converte_data("data_fundacao")) 
 df = df.withColumn("data_listagem", converte_data("data_listagem"))
 display(df)


In [0]:
#Comparar resultado com a query 11
for c in ["data_fundacao", "data_listagem"]:
    print(c, "| nulos depois:", df.filter(F.col(c).isNull()).count())

df.select("data_fundacao", "data_listagem").printSchema()   
# esperado: date

In [0]:
#Trabalhando com a coluna valor de mercado
# Referência para a reconciliação do valor_mercado 
print("nulos antes:", df.filter(F.col("valor_mercado").isNull()).count())                  
print("formato BR (R$):", df.filter(F.col("valor_mercado").startswith("R$")).count())
print("formato simples:", df.filter(~F.col("valor_mercado").startswith("R$")).count())

In [0]:
#Tratando os dados de valor de mercado
#Primeiro tratando os dados formato BR que começam com R$ (eles tem a vírgula como separador decimal, ao contrário do formato simples que usa ponto)
#Para dados que começam com R$, Tirar o R$ depois eliminar todos os pontos e depois trocar a vírgula que sobrou por ponto

vm = F.col("valor_mercado")

# Só as linhas BR passam pela limpeza; o formato simples (ponto decimal) fica intacto
df = df.withColumn(
    "valor_mercado",
    F.when(
        vm.startswith("R$"),
        F.regexp_replace(
            F.regexp_replace(
                F.regexp_replace(vm, r"R\$\s*", ""),
                r"\.", ""),
            ",", ".")
    ).otherwise(vm)
)

# try_cast devolve NULL em vez de derrubar a célula; double por causa da precisão
df = df.withColumn("valor_mercado", F.col("valor_mercado").try_cast("double"))




In [0]:
df.select("valor_mercado").printSchema()                                         # esperado: double
print("nulos depois:", df.filter(F.col("valor_mercado").isNull()).count())       # esperado: 40
df.agg(F.min("valor_mercado"), F.max("valor_mercado")).show()                    # casa dos bilhões
print("abaixo de 1 milhão:", df.filter(F.col("valor_mercado") < 1_000_000).count())  # esperado: perto de 0

In [0]:
#Tratar número de funcionário. Nesse caso o ponto sempre será separador da casa do milhar 
#Tirar os pontos
df = df.withColumn("num_funcionarios", F.regexp_replace(F.col("num_funcionarios"), r"\.", ""))

#Transformar os valores negativos para dados nulos 
df = df.withColumn("num_funcionarios", F.when(F.col("num_funcionarios") < 0, None).otherwise(F.col("num_funcionarios")))

#Converter tudo para integer
df = df.withColumn("num_funcionarios", F.col("num_funcionarios").cast("integer"))



In [0]:
df.select("num_funcionarios").printSchema()                                         # esperado: integer
print("nulos depois:", df.filter(F.col("num_funcionarios").isNull()).count())       # esperado um pouco mais de 40 (nulos + os negativos)
df.agg(F.min("num_funcionarios"), F.max("num_funcionarios")).show()

In [0]:
# CNPJ: deixa só os dígitos e marca quais têm 14 (tamanho de um CNPJ válido).
# Inválidos ficam marcados ma coluna cnpj_valido = False em vez de apagados, para preservar a informação e investigar no futuro
df = df.withColumn("cnpj", F.regexp_replace(F.col("cnpj"), r"[^0-9]", ""))
df = df.withColumn("cnpj_valido", F.length(F.col("cnpj")) == 14)


In [0]:

#Conferir se a limpeza funcionou

ref = bronze.filter(F.length(F.regexp_replace(F.trim("cnpj"), r"[^0-9]", "")) != 14).count()
print("CNPJs sem 14 dígitos na bronze:", ref)
print("inválidos no df:", df.filter(~F.col("cnpj_valido")).count())   # esperado: igual ao de cima
print("válidos no df:", df.filter(F.col("cnpj_valido")).count())      # inválidos + válidos = 476
df.select(F.length("cnpj").alias("digitos")).groupBy("digitos").count().show()   # esperado: só 14 e 13

In [0]:
# Tratar Telefone. Chegam em 3 formatos: "(11) 91234-5678", "11912345678" e "+55 11 91234-5678".
# Padroniza telefone: remove caracteres não numéricos
tel = F.regexp_replace(F.col("telefone"), r"[^0-9]", "")

# Se começar com 55 e tiver 13 dígitos, remove o cód do país e fica com 11 dígitos
tel = F.when((F.length(tel) == 13) & tel.startswith("55"), tel.substr(3, 11)).otherwise(tel)

# Atualiza a coluna com o telefone padronizado
df = df.withColumn("telefone", tel)

# Válido = 11 dígitos. Se o telefone for NULL, telefone_valido também é NULL
df = df.withColumn("telefone_valido", F.length(F.col("telefone")) == 11)



In [0]:
# Transformar E-MAIL e SITE em minúsculas.  
df = df.withColumn("email_ri", F.lower(F.col("email_ri")))
df = df.withColumn("site", F.lower(F.col("site")))

# Criar coluna e-mail_valido com uma regex simples (texto + "@" + texto + "." + texto, sem espaços). Checar se segue esse padrão
df = df.withColumn("email_valido", F.col("email_ri").rlike(r"^[^@\s]+@[^@\s]+\.[^@\s]+$"))

In [0]:
# TELEFONE: todos com 11 dígitos, exceto os nulos (não informado)
df.select(F.length("telefone").alias("digitos")).groupBy("digitos").count().show()   # esperado: 11 e nulo
print("telefones inválidos:", df.filter(F.col("telefone_valido") == False).count())  # esperado: 0
print("nulos telefone:", df.filter(F.col("telefone").isNull()).count())              # esperado: 129

# E-MAIL: nenhum com maiúscula, nenhum formato inválido, nulos iguais aos de antes
print("e-mails com maiúscula:", df.filter(F.col("email_ri") != F.lower(F.col("email_ri"))).count())  # esperado: 0
print("e-mails inválidos:", df.filter(F.col("email_valido") == False).count())                       # esperado: 0
print("nulos e-mail:", df.filter(F.col("email_ri").isNull()).count())                                # esperado: 65

# SITE: nulos iguais aos de antes e nenhum com maiúscula
print("nulos site:", df.filter(F.col("site").isNull()).count())                                      # esperado: 106
print("sites com maiúscula:", df.filter(F.col("site") != F.lower(F.col("site"))).count())            # esperado: 0

In [0]:
display(df)

In [0]:
# Tratando dados de id_empresa. Removendo duplicidades
# w = separe por id_empresa (criar grupo) e dentro de cada empresa ordene do registro mais recente para o mais antigo (desc)
w = Window.partitionBy("id_empresa").orderBy(F.col("_ingestion_ts").desc())

# Deduplicação. Versões mais antigas descartadas, mantive o registro mais recente 
df = df.withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn")

In [0]:
print("linhas:", df.count(), "| ids distintos:", df.select("id_empresa").distinct().count())   # esperado números iguais
print("ids repetidos:", df.groupBy("id_empresa").count().filter("count > 1").count())          # esperado: 0
print("listagem antes da fundação:", df.filter(F.col("data_listagem") < F.col("data_fundacao")).count())  # esperado: 0
df.printSchema()

In [0]:
# Salvar a tabela
df.write.mode("overwrite").saveAsTable("workspace.silver.empresas")

In [0]:
# Verificando se a tabela foi salva
print(spark.table("workspace.silver.empresas").count())   # esperado: 450